# 선형회귀 기준모델
ZIP의 원본 분석 흐름을 저장소 경로에 맞게 정리한 공개용 사본입니다.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error


In [ ]:
target_col = 'Book-Rating'
text_cols = ['Book-Title', 'Book-Author', 'Publisher', 'Location_country']
num_cols = ['Age', 'Year-Of-Publication']
df = pd.read_csv('../data/processed/df_final1.csv')
X = df[text_cols + num_cols].copy()
y = df[target_col].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
def tfidf_svd(n_components, ngram_range=(1, 2), max_features=50000):
    return Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)),
        ('svd', TruncatedSVD(n_components=n_components, random_state=42)),
    ])
preprocess = ColumnTransformer([
    ('title', tfidf_svd(500), 'Book-Title'),
    ('author', tfidf_svd(80, (1, 1)), 'Book-Author'),
    ('publisher', tfidf_svd(50, (1, 1)), 'Publisher'),
    ('loc', tfidf_svd(10, (1, 1)), 'Location_country'),
    ('num', StandardScaler(), num_cols),
])
model = Pipeline([('prep', preprocess), ('lr', LinearRegression())])


In [ ]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, scoring='neg_root_mean_squared_error', cv=cv, n_jobs=-1)
print('CV RMSE:', -scores.mean())
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Test RMSE:', root_mean_squared_error(y_test, pred))
print('Test MAE:', mean_absolute_error(y_test, pred))
print('Test R2:', r2_score(y_test, pred))
